In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. List all sector source files
sector_files = [
    "Banks.csv",
    "Capital Goods.csv",
    "Consumer Durables & Apparel.csv",
    "Consumer Services.csv",
    "Diversified Financials.csv",
    "Energy.csv",
    "Food, Beverage & Tobacco.csv",
    "Materials.csv",
    "Real Estate Management & Development.csv",
    "Retailing.csv",
    "Software & Services.csv",
    "Telecommunication Services.csv",
    "Transportation.csv",
    "Utilities.csv",
    "Insurance.csv",
    "Automobiles & Components.csv",
    "Healthcare Equipment & Services.csv",
    "Commercial & Professional services.csv",
    "Food & Staples Retailing.csv",
    "Household & Personal Products.csv",
]

# 2. Read each sector file and build one merged dataset
base_dir = Path(r"c:/Files/Uni/S4/DS/Project Dataset/Raw Data")
master = None

for name in sector_files:
    file_path = base_dir / name
    sector_df = pd.read_csv(file_path)
    sector_df.columns = [c.strip() for c in sector_df.columns]

    if "Traded Date" not in sector_df.columns or "Value" not in sector_df.columns:
        raise ValueError(f"Unexpected columns in {name}: {sector_df.columns.tolist()}")

    out_col = Path(name).stem.replace(" ", "_").replace("&", "and")

    s = sector_df[["Traded Date", "Value"]].copy()
    s["date"] = pd.to_datetime(s["Traded Date"], format="%d %b %Y", errors="coerce")
    s[out_col] = pd.to_numeric(s["Value"], errors="coerce")
    s = s[["date", out_col]].dropna(subset=["date"]).sort_values("date")

    if master is None:
        master = s
    else:
        master = master.merge(s, on="date", how="outer")

master = master.sort_values("date").set_index("date")
all_sectors = master.columns

# 3. Compute Zero-Return Ratio (illiquidity) using merged data
print("--- Zero-Return Ratio (Illiquidity) Report ---")
results = []

for col in all_sectors:
    series = master[col]
    valid_prices = series.where(series > 0)
    returns = np.log(valid_prices / valid_prices.shift(1))

    total_days = returns.notna().sum()
    zero_days = returns.eq(0.0).sum()
    zrr = (zero_days / total_days) * 100 if total_days > 0 else np.nan

    results.append(
        {
            "Sector": col,
            "Zero_Days": int(zero_days),
            "Total_Days": int(total_days),
            "ZRR (%)": round(zrr, 2) if pd.notna(zrr) else np.nan,
        }
    )

# 4. Display and save report
zrr_df = pd.DataFrame(results).sort_values(by="ZRR (%)", ascending=False)
print(zrr_df.to_string(index=False))
zrr_df.to_csv("sector_liquidity_report.csv", index=False)

--- Zero-Return Ratio (Illiquidity) Report ---
                                Sector  Zero_Days  Total_Days  ZRR (%)
       Household_and_Personal_Products        340        1638    20.76
            Automobiles_and_Components        210        1638    12.82
  Commercial_and_Professional_services        195        1640    11.89
                        Transportation        188        1640    11.46
            Food_and_Staples_Retailing        107        1640     6.52
                 Software_and_Services         33         956     3.45
            Telecommunication_Services         45        1638     2.75
                                Energy          9        1640     0.55
                     Consumer_Services          3        1640     0.18
                             Retailing          2        1638     0.12
                                 Banks          2        1640     0.12
                             Insurance          2        1640     0.12
                             M

In [15]:
import pandas as pd
import numpy as np

# Build log returns from the merged level data in `master`
returns_df = np.log(master / master.shift(1))

# Select sectors and compute their return correlation
selected = ["Insurance", "Banks", "Diversified_Financials"]
corr_matrix = returns_df[selected].dropna(how="any").corr()

print(corr_matrix)

                        Insurance     Banks  Diversified_Financials
Insurance                1.000000  0.175102                0.226256
Banks                    0.175102  1.000000                0.484257
Diversified_Financials   0.226256  0.484257                1.000000
